# Orientation-Cluster Routed Patch Upsampler Design

This notebook specifies the proposed **Orientation-Cluster Routed Patch Upsampler**
(short name: **OCRP Upsampler**) for quaternion SR. The design replaces
parent-dominated per-token semiglobal routing with:

1. label-free local orientation clustering on the LR quaternion bank,
2. deterministic cluster-slot ordering,
3. cheap per-slot metadata,
4. representative equivariant slot extraction via medoid selection,
5. joint routing over the full `4x4` HR patch,
6. discrete cross-slot ownership at the patch output.

The goal is to preserve distinct local orientation hypotheses, reduce parent-feature
bias near boundaries, and avoid the NN-style artifacts that arise when each HR pixel
is routed independently.


## Design Commitments

- **Cluster source**: the local `5x5` LR quaternion bank is clustered directly from
  quaternions, not from pooled feature summaries.
- **Cluster threshold**: start with symmetry-aware `4`-neighbor connectivity and a
  default boundary threshold of `2 deg`.
- **Slot semantics**:
  - `slot 1 = cluster containing the parent LR pixel`
  - `slot 2 = strongest non-parent cluster by mass`
  - `slot 3 = next strongest non-parent cluster`
  - `slot 4 = null / unused`
- **Metadata stays cheap**:
  - `valid_k`
  - `slot_type_k`
  - `mass_k`
  - `centroid_k`
  - optional `spatial_dispersion_k`
- **Parent feature is a weak continuity prior**, not the dominant routing signal.
- **Within-slot context building uses medoid selection**; **across-slot ownership is discrete**.
- **Routing is patchwise** over the whole `4x4` HR patch, not a collection of `16`
  disjoint per-pixel routing decisions.
- **Start with one shared patch proposal head**. Only add per-slot or per-expert heads
  after slot semantics are stable and verified.
- **The patch proposal head is not a plain MLP**. It must stay consistent with the
  repo formulation: invariant routing and scalar control are allowed, but feature
  synthesis itself must be done with an **equivariant e3nn TP-based head**.


In [7]:
DESIGN_DEFAULTS = {
    "window_size": 5,
    "upsample_factor": 4,
    "hr_patch_tokens": 16,  # 4x4
    "kmax_slots": 4,
    "cluster_connectivity": 4,
    "cluster_threshold_deg": 2.0,
    "slot_order": [
        "parent_cluster",
        "primary_nonparent",
        "secondary_nonparent",
        "null",
    ],
    "metadata_fields": [
        "valid_k",
        "slot_type_k",
        "mass_k",
        "centroid_y_k",
        "centroid_x_k",
        "optional_spatial_dispersion_k",
    ],
    "within_slot_context_builder": "medoid_selection",
    "cross_slot_ownership": "hard_discrete",
    "proposal_head": "shared_patch_proposal_head",
}

DESIGN_DEFAULTS


{'window_size': 5,
 'upsample_factor': 4,
 'hr_patch_tokens': 16,
 'kmax_slots': 4,
 'cluster_connectivity': 4,
 'cluster_threshold_deg': 2.0,
 'slot_order': ['parent_cluster',
  'primary_nonparent',
  'secondary_nonparent',
  'null'],
 'metadata_fields': ['valid_k',
  'slot_type_k',
  'mass_k',
  'centroid_y_k',
  'centroid_x_k',
  'optional_spatial_dispersion_k'],
 'within_slot_context_builder': 'medoid_selection',
 'cross_slot_ownership': 'hard_discrete',
 'proposal_head': 'shared_patch_proposal_head'}

## System Architecture

```text
LR quaternions
    |
    v
Local-iso encoder
    |
    v
LR feature map
    |
    +--> For each LR parent cell:
          |
          +--> build 5x5 quaternion bank + 5x5 feature bank
          |
          +--> cluster the quaternion bank (label-free)
          |
          +--> order clusters into slots: parent / alt1 / alt2 / null
          |
          +--> compute cheap slot metadata
          |
          +--> medoid selection inside each slot
          |     -> representative equivariant slot contexts c_1 ... c_K
          |
          +--> patch router predicts slot logits for the whole 4x4 HR patch jointly
          |     -> ownership label map over slots
          |
          +--> shared patch proposal head applied to each slot context
          |     -> K slot-conditioned HR patch proposals
          |
          +--> hard gather/select using ownership labels
                -> one HR feature patch
    |
    v
stitch HR feature patches into HR feature map
    |
    v
quaternion decoder
    |
    v
SR quaternions
```

Key principle: **no final soft mixing across orientation clusters**. Training may use a
relaxation for gradients, but the modeled ownership is discrete.


## Dataflow And Tensor Shapes

| Symbol | Meaning | Suggested shape |
| --- | --- | --- |
| `q_lr` | LR quaternions | `(B, H*W, 4)` |
| `f_lr` | LR encoded features | `(B, H*W, C)` |
| `bank_q` | local quaternion bank per parent | `(B, H*W, 25, 4)` |
| `bank_f` | local feature bank per parent | `(B, H*W, 25, C)` |
| `cluster_mask` | cluster membership before slot packing | variable or `(B, H*W, Kmax, 25)` |
| `slot_mask` | slot membership mask | `(B, H*W, 4, 25)` |
| `slot_valid` | slot validity | `(B, H*W, 4)` |
| `slot_meta` | cheap metadata per slot | `(B, H*W, 4, D_meta)` |
| `slot_ctx` | representative equivariant context per slot | `(B, H*W, 4, C)` |
| `phase_grid` | 4x4 HR phase positions inside one parent | `(16, D_phase)` or `(B, H*W, 16, D_phase)` |
| `router_logits` | slot logits for every HR token in the patch | `(B, H*W, 16, 4)` |
| `owner_idx` | discrete slot label for each HR token | `(B, H*W, 16)` |
| `patch_prop` | per-slot HR patch proposal | `(B, H*W, 4, 16, C)` |
| `patch_out` | selected HR feature patch | `(B, H*W, 16, C)` |

Cheap metadata recommendation:

```text
meta_k = [
    valid_k,
    onehot(slot_type_k),
    mass_k,
    centroid_y_k,
    centroid_x_k,
    optional spatial_dispersion_k,
]
```


## Current RR-CTP Router vs Proposed Slot Router

### Current RR-CTP router

The current semiglobal RR-CTP upsampler builds a `5x5` candidate bank, gates it,
forms expert-specific soft contexts, then routes using a coarse summary of the gated
bank together with the phase-conditioned parent query. In practice that means the
router mostly decides:

> given the parent-derived query and a whole-window summary, which expert family should win?

This is effective, but it still lets parent bias leak deeply into the routing stack,
and it does not explicitly represent distinct local orientation hypotheses.

### Proposed slot router

The new router scores **cluster slots** directly. It should decide:

> for each HR token in the `4x4` patch, which local orientation cluster should own it?

The router sees:

- the slot context `c_k`,
- the simple metadata `meta_k`,
- the HR phase token,
- optionally a weak parent prior or a patch summary.

The parent cluster remains useful as a continuity prior, but it should not dominate the
decision near boundaries.


## What Each Step In The Upsampling Stage Does

### Step 1. Build the local LR banks

- Extract the `5x5` quaternion bank and `5x5` feature bank around each parent LR pixel.
- Purpose: expose all locally relevant orientation and feature candidates before any pooling.
- Output: `bank_q`, `bank_f`.

### Step 2. Cluster the quaternion bank

- Use symmetry-aware neighbor misorientation with a small threshold, starting from
  `2 deg`, and `4`-neighbor connectivity inside the `5x5` bank.
- Purpose: split the local bank into explicit orientation hypotheses without LR labels.
- Output: raw cluster ids over the `25` bank positions.

### Step 3. Pack clusters into deterministic slots

- Assign the parent-containing cluster to `slot 1`.
- Assign the strongest non-parent cluster to `slot 2`.
- Assign the next strongest non-parent cluster to `slot 3`.
- Use `slot 4` as null.
- Purpose: make slot meaning stable enough for routing and later specialization.
- Output: `slot_mask`, `slot_valid`, `slot_type`.

### Step 4. Compute cheap slot metadata

- Compute only cheap descriptors from membership and positions.
- Recommended fields: `valid_k`, `slot_type_k`, `mass_k`, `centroid_k`, and optional
  `spatial_dispersion_k`.
- Purpose: tell the router what each slot is, without expensive extra geometry.
- Output: `slot_meta`.

### Step 5. Build one representative equivariant slot context by medoid selection

- Work **within** each real slot only; do not average across slots.
- Select the medoid member of the slot and use its equivariant feature as the slot context.
- Purpose: keep the slot representative pure, avoid re-mixing already-separated orientations,
  and preserve compatibility with the equivariant formulation.
- Output: `slot_ctx`.

### Step 6. Form the joint patch-routing query

- Build the `4x4` HR phase tokens for the current parent cell.
- Optionally include a weak parent prior or a small patch summary.
- Purpose: let all `16` HR tokens be routed together instead of as isolated pixels.
- Output: per-patch router input state.

### Step 7. Predict discrete slot ownership for the full `4x4` patch

- The router outputs slot logits for all `16` HR tokens jointly.
- Invalid slots are masked before normalization.
- Purpose: decide which cluster should own each HR token while enforcing patch-level coherence.
- Output: `router_logits`, then `owner_idx` or one-hot ownership masks.

### Step 8. Produce per-slot patch proposals with a shared head

- Apply one shared patch proposal head to each slot context.
- Purpose: generate what the HR patch would look like if that slot owned it.
- Output: `patch_prop` with one proposal per slot.

### Step 9. Assemble the final HR feature patch by hard selection

- Gather the proposal value from the selected slot at each HR token.
- Purpose: keep cross-slot ownership discrete instead of blending incompatible orientation modes.
- Output: `patch_out`.

### Step 10. Stitch patches and decode

- Rearrange the selected `4x4` patches into the global HR feature map.
- Decode HR features back to quaternions.
- Purpose: complete SR while preserving coherent ownership decisions made in the patch router.


## Step-By-Step Implementation Plan

### Phase A. Bank extraction and clustering helper

1. Reuse or adapt the existing `5x5` bank extraction utilities from RR-CTP.
2. Add a quaternion-bank clustering helper that returns cluster ids per parent window.
3. Verify that the cluster helper is label-free and deterministic.

### Phase B. Slot builder and metadata

1. Implement parent / alt1 / alt2 / null slot packing.
2. Add `valid_k`, `slot_type_k`, `mass_k`, `centroid_k`, and optional `spatial_dispersion_k`.
3. Verify that `slot 1` always contains the parent cluster when one exists.

### Phase C. Medoid representative slot extraction

1. Start with medoid selection inside each real slot.
2. Mask null slots cleanly.
3. Verify that medoid slot contexts preserve cluster identity better than any within-slot averaging.

### Phase D. Patch router and shared TP-based proposal head

1. Replace per-token expert routing with joint `4x4` patch routing.
2. Output patchwise slot logits with invalid-slot masking.
3. Add the shared **TP-based equivariant** patch proposal head and hard gather logic.

### Phase E. Training relaxation and stability

1. Use argmax at inference.
2. During training, use a straight-through estimator or annealed Gumbel-softmax.
3. Add light regularization to discourage checkerboard ownership maps.

### Phase F. Debugging and visualization

1. Visualize cluster maps in the `5x5` bank.
2. Visualize slot metadata and slot validity.
3. Visualize `4x4` ownership labels and final assembled HR patches.
4. Compare boundary strips against the current RR-CTP walkthrough notebook.


## Nuances Locked In

The following choices are part of the design and should not drift during implementation:

- **Parent bias**: the parent LR feature is a weak continuity prior only. It can help in
  clean interiors, but it must not dominate the final slot decision near boundaries.
- **Patchwise routing**: routing is not a collection of `16` disjoint decisions. The router
  predicts ownership for the full `4x4` patch jointly so neighboring HR pixels can influence
  each other and the SR patch stays globally coherent.
- **Cheap metadata only**: metadata should remain simple. Use `valid_k`, `slot_type_k`,
  `mass_k`, `centroid_k`, and optional `spatial_dispersion_k`. Do not add expensive extra
  geometric descriptors in the first version.
- **No within-slot re-mixing of orientation modes**: clustering and slotting already separate
  orientation hypotheses. Step 5 should only choose a representative from the slot, not try
  to solve orientation separation again.
- **Medoid slot context**: build one representative equivariant slot context from the
  already-separated slot, using medoid selection.
- **Hard cross-slot ownership**: the final HR patch is assembled by discrete slot selection,
  not soft averaging across incompatible clusters.
- **Shared-head first**: the first version is not a compute-sparse MoE. It is a dense,
  routed, multi-hypothesis patch upsampler with stable slot semantics and one shared compute
  head.
- **Equivariant compute path**: the compute head that turns the chosen slot into HR features
  must remain consistent with the repo formulation. Routing may use invariant statistics, but
  feature synthesis must be done by an **e3nn TP-based equivariant head**, not by a plain MLP.


## Explicit Implementation Notes

### 1. Local bank extraction and clustering

For each parent LR pixel:

1. build `bank_q` and `bank_f` of shape `(25, 4)` and `(25, C)`
2. compute symmetry-aware `4`-neighbor misorientation on the `5x5` bank
3. mark boundaries where misorientation `> 2 deg`
4. run connected components on the remaining graph
5. assign connected components to slots: parent / alt1 / alt2 / null

This produces `slot_mask`, `slot_valid`, `slot_type`, and the cheap metadata.

### 2. Medoid slot context builder

For each real slot `k`, do **medoid selection** rather than averaging.

Recommended implementation:

1. collect all bank members that belong to slot `k`
2. compute the in-slot pairwise symmetry-aware misorientation matrix
3. choose the medoid index `m_k` that minimizes summed in-slot misorientation
4. set the representative slot context `c_k = bank_f[m_k]`

This stays cheap because each slot comes from a `5x5` bank and has at most `25` members.
The representative context is still an equivariant feature because it is one of the original
bank features, not an arbitrary scalar projection.

### 3. Router input and output

The router should operate in invariant / scalar space. It sees:

- `slot_ctx` through invariant summaries
- `slot_meta`
- `phase_grid`
- optional weak parent prior

Implementation target:

- input: `(B, H*W, Kmax, C)` slot contexts, `(B, H*W, Kmax, D_meta)` metadata,
  `(16, D_phase)` or `(B, H*W, 16, D_phase)` phase tokens
- output: `router_logits` of shape `(B, H*W, 16, Kmax)`
- invalid slots are masked before softmax or argmax

### 4. Shared TP-based equivariant proposal head

The proposal head is where the actual HR feature computation happens.

It should be implemented in the same language as the rest of the repo:

- equivariant features live in `irreps_feat`
- metadata and phase only act as scalar controls
- actual feature synthesis uses `IrrepsLinear` and `FullyConnectedTensorProduct`

A good starting structure is:

1. build a weak equivariant patch query from the phase grid and optional weak parent prior
2. lift slot medoid context `c_k` with `IrrepsLinear`
3. lift the patch query with `IrrepsLinear`
4. mix query and slot context with `FullyConnectedTensorProduct`
5. use scalar controls from phase / metadata to gate or rescale block outputs
6. output one HR patch proposal `P_k` of shape `(16, C)` for each slot

Important: the head should not be a plain concatenation MLP from `[slot_ctx, meta, phase]`
to features. The scalar controls are allowed, but the equivariant feature path must stay TP-based.

### 5. Step 7 vs Step 8 vs Step 9

These three steps must stay conceptually separate:

- **Step 7**: the router predicts **ownership**
  - output: `router_logits`, then `owner_idx`
  - meaning: which slot should own each HR token?
- **Step 8**: the shared TP-based head predicts **slot-conditioned HR patch proposals**
  - output: `patch_prop[k, t, :]`
  - meaning: what would token `t` look like if slot `k` owned it?
- **Step 9**: hard **gather / assembly**
  - output: `patch_out[t, :] = patch_prop[owner_idx[t], t, :]`
  - meaning: use the chosen slot's proposal at each HR token

So Step 7 makes the decision, Step 8 builds the candidate equivariant outputs, and
Step 9 performs the hard selection that assembles the final HR patch.

### 6. Training-time discrete ownership

Inference target:

```text
owner_idx = argmax(router_logits, dim=-1)
patch_out[t, :] = patch_prop[owner_idx[t], t, :]
```

Training relaxation:

- use straight-through argmax or annealed Gumbel-softmax for the ownership map
- keep the modeled semantics discrete even if training uses a surrogate gradient

### 7. Version-1 interpretation

Version 1 should be viewed as:

- a **slot-routed, multi-hypothesis patch upsampler**
- not primarily as a compute-saving MoE
- with specialization carried first by the slot semantics and the medoid contexts,
  not by separate per-slot networks


In [8]:
class QuaternionBankClusterer:
    """Cluster a local 5x5 LR quaternion bank into orientation-connected components."""

    def __call__(self, bank_q):
        raise NotImplementedError


class ClusterSlotBuilder:
    """Pack clusters into parent / alt1 / alt2 / null slots and emit cheap metadata."""

    def __call__(self, cluster_ids, parent_bank_index):
        raise NotImplementedError


class MedoidSlotContextBuilder:
    """Build one representative equivariant slot context per slot using medoid selection."""

    def __call__(self, bank_q, bank_f, slot_mask):
        raise NotImplementedError


class PatchSlotRouter:
    """Predict joint 4x4 slot ownership logits for the full HR patch."""

    def __call__(self, slot_ctx, slot_meta, phase_grid, weak_parent_prior=None):
        raise NotImplementedError


class SharedTPPatchProposalHead:
    """Produce one HR patch proposal per slot using TP-based equivariant mixing."""

    def __call__(self, slot_ctx, slot_meta, phase_grid, weak_parent_prior=None):
        raise NotImplementedError


class OrientationClusterRoutedPatchUpsampler:
    """High-level OCRP upsampler composition."""

    def __init__(self):
        self.clusterer = QuaternionBankClusterer()
        self.slot_builder = ClusterSlotBuilder()
        self.context_builder = MedoidSlotContextBuilder()
        self.router = PatchSlotRouter()
        self.head = SharedTPPatchProposalHead()

    def forward(self, bank_q, bank_f, parent_bank_index, phase_grid, weak_parent_prior=None):
        raise NotImplementedError


## Rules To Preserve During Implementation

- Do not let parent-derived features dominate boundary decisions.
- Keep slot metadata simple and cheap.
- Keep cross-slot ownership discrete.
- Let the router make patchwise decisions jointly for the `4x4` HR patch.
- Keep the first version interpretable: stable slot semantics, shared TP-based proposal head,
  explicit validity masking, and simple debugging outputs.
- Treat this as a **dense routed multi-hypothesis patch upsampler**, not primarily as
  a compute-saving MoE.
- Preserve the separation of concerns:
  - Step 7 decides ownership
  - Step 8 computes TP-based equivariant slot proposals
  - Step 9 gathers the selected proposal into the final HR patch
